# Streaming Data Loading Memory Test

**Purpose:** Test the streaming implementation and monitor memory usage to verify:
- No eager loading of query segments
- Low peak memory usage
- Correct HDF5 handle behavior
- Proper lazy loading through DataLoader

## Prerequisites

**Required Files:**
- CSV metadata files (e.g., `eval_dir/*.csv`)
- WAV audio files (e.g., `eval_dir/*.wav`)
- HDF5 feature files (e.g., `eval_dir/*_pcen.h5`)
- Frequency masks (optional, in `mask_dir/mask_*.h5` or `mask_*.npy`)

**Update the paths below before running:**

In [ ]:
# ============================================================================
# USER CONFIGURATION - CHOOSE ONE OPTION
# ============================================================================

# OPTION 1: Load from YAML config file (recommended)
USE_YAML_CONFIG = True
YAML_CONFIG_PATH = "configs/test.yaml"  # Path relative to baselines/dcase2024_task5/

# OPTION 2: Manual configuration (if USE_YAML_CONFIG = False)
# Path to evaluation directory containing CSV, WAV, and HDF5 files
EVAL_DIR = "/path/to/eval_dir"  # e.g., "/data/dcase2024/Development_Set/Validation_Set"

# Path to mask directory (optional, set to None if not using)
MASK_DIR = "/path/to/mask_dir"  # e.g., "/data/dcase2024/masks" or None

# Test directory (optional, set to None if not using)
TEST_DIR = None  # e.g., "/data/dcase2024/Evaluation_Set" or None

# Feature configuration
FEATURE_TYPES = "pcen"  # Feature suffix (e.g., "pcen" for *_pcen.h5)
SR = 22050  # Sample rate
HOP_MEL = 256  # Hop length for mel spectrogram

# Training parameters
N_SHOT = 5  # Number of support examples

# Evaluation parameters
NEGATIVE_ESTIMATE = "freq_mask"  # "freq_mask" or "rms"
TEST_SEGLEN_LEN_LIM = 128
TEST_HOPLEN_FENMU = 4

## Section 1: Environment Setup

In [ ]:
import sys
import os
import gc
import tracemalloc
import psutil
import numpy as np
import torch
from torch.utils.data import DataLoader
from types import SimpleNamespace

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import project modules
from baselines.dcase2024_task5.src.datamodules.components.test_loader_ada_seglen_better_neg_v2 import (
    PrototypeAdaSeglenBetterNegTestSetV2
)
from baselines.dcase2024_task5.src.datamodules.components.streaming_segment_dataset import (
    StreamingSegmentDataset
)

print("✓ Imports successful")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

### Helper Functions

In [ ]:
def print_mem(label=""):
    """Print current memory usage in MB."""
    process = psutil.Process(os.getpid())
    mem_mb = process.memory_info().rss / 1024 / 1024
    print(f"[{label}] Memory: {mem_mb:.2f} MB")
    return mem_mb

def describe_batch(batch):
    """Print structure of batch for debugging."""
    if isinstance(batch, tuple):
        print(f"Batch is tuple with {len(batch)} elements:")
        for i, item in enumerate(batch):
            describe_item(item, indent=f"  [{i}]")
    else:
        describe_item(batch, indent="")

def describe_item(item, indent=""):
    """Recursively describe item structure."""
    if isinstance(item, (np.ndarray, torch.Tensor)):
        dtype = item.dtype
        shape = item.shape
        mem_mb = item.nbytes / 1024 / 1024 if isinstance(item, np.ndarray) else item.element_size() * item.nelement() / 1024 / 1024
        print(f"{indent} Array: shape={shape}, dtype={dtype}, mem={mem_mb:.2f}MB")
    elif isinstance(item, list):
        print(f"{indent} List with {len(item)} items")
        if len(item) > 0:
            print(f"{indent}   First item type: {type(item[0])}")
            if isinstance(item[0], dict):
                print(f"{indent}   First item keys: {list(item[0].keys())}")
    elif isinstance(item, dict):
        print(f"{indent} Dict with keys: {list(item.keys())}")
    elif isinstance(item, tuple):
        print(f"{indent} Tuple with {len(item)} elements")
        for i, sub_item in enumerate(item):
            describe_item(sub_item, indent=f"{indent}  [{i}]")
    else:
        print(f"{indent} {type(item).__name__}: {item if not hasattr(item, '__len__') or len(str(item)) < 50 else str(item)[:50] + '...'}")

def force_gc():
    """Force garbage collection and print memory."""
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return print_mem("After GC")

print("✓ Helper functions defined")

### Create Configuration Objects

In [ ]:
# Load configuration
if USE_YAML_CONFIG:
    print("Loading configuration from YAML file...")
    from omegaconf import OmegaConf
    
    # Construct full path to config file
    config_path = os.path.join(os.getcwd(), YAML_CONFIG_PATH)
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")
    
    # Load YAML config
    cfg = OmegaConf.load(config_path)
    
    # Extract configuration sections
    path_config = cfg.path
    features_config = cfg.features
    train_param_config = cfg.train_param
    eval_param_config = cfg.eval_param
    
    print("✓ Configuration loaded from YAML")
    print(f"  Config file: {config_path}")
    print(f"  Eval dir: {path_config.eval_dir}")
    print(f"  Test dir: {path_config.test_dir}")
    print(f"  Mask dir: {path_config.mask_dir}")
    print(f"  Feature types: {features_config.feature_types}")
    print(f"  N-shot: {train_param_config.n_shot}")
    print(f"  Negative estimate: {eval_param_config.negative_estimate}")
    print(f"  Test seglen limit: {features_config.test_seglen_len_lim}")
    print(f"  Test hoplen fenmu: {features_config.test_hoplen_fenmu}")
    
    # Display full config structure
    print("\n=== Full Configuration ===")
    print(OmegaConf.to_yaml(cfg))
    
else:
    print("Using manual configuration...")
    # Create configuration objects using SimpleNamespace
    path_config = SimpleNamespace(
        eval_dir=EVAL_DIR,
        test_dir=TEST_DIR,
        mask_dir=MASK_DIR
    )
    
    features_config = SimpleNamespace(
        feature_types=FEATURE_TYPES,
        sr=SR,
        hop_mel=HOP_MEL,
        test_seglen_len_lim=TEST_SEGLEN_LEN_LIM,
        test_hoplen_fenmu=TEST_HOPLEN_FENMU
    )
    
    train_param_config = SimpleNamespace(
        n_shot=N_SHOT
    )
    
    eval_param_config = SimpleNamespace(
        negative_estimate=NEGATIVE_ESTIMATE
    )
    
    print("✓ Configuration objects created")
    print(f"  Eval dir: {path_config.eval_dir}")
    print(f"  Mask dir: {path_config.mask_dir}")
    print(f"  N-shot: {train_param_config.n_shot}")

## Section 2: Inspect Dataset

Instantiate the dataset and examine its structure without loading data.

In [ ]:
# Start memory tracking
tracemalloc.start()
baseline_mem = print_mem("Baseline")

# Instantiate dataset
print("\nInstantiating dataset...")
dataset = PrototypeAdaSeglenBetterNegTestSetV2(
    path=path_config,
    features=features_config,
    train_param=train_param_config,
    eval_param=eval_param_config
)

after_init_mem = print_mem("After dataset init")
print(f"Memory increase: {after_init_mem - baseline_mem:.2f} MB")

# Print dataset info
print(f"\nDataset length: {len(dataset)}")
print(f"CSV files found: {len(dataset.all_csv_files)}")
if len(dataset.all_csv_files) > 0:
    print(f"First CSV: {os.path.basename(dataset.all_csv_files[0])}")
    print(f"Last CSV: {os.path.basename(dataset.all_csv_files[-1])}")

print("\n✓ Dataset instantiated successfully")
print("⚠️  Note: No data loaded yet (lazy loading)")

## Section 3: Test `__getitem__()`

Load a single sample and verify:
- Support segments are numpy arrays (eager-loaded)
- Query segments are metadata (lazy-loaded)
- HDF5 handles are returned

In [ ]:
print("Loading first sample...")
before_getitem_mem = print_mem("Before __getitem__")

# Load sample
sample = dataset[0]

after_getitem_mem = print_mem("After __getitem__")
print(f"Memory increase: {after_getitem_mem - before_getitem_mem:.2f} MB")

# Unpack sample
(
    (
        X_pos,
        X_neg,
        X_query_info,
        X_pos_neg,
        X_neg_neg,
        X_query_info_neg,
        hop_seg,
        hop_seg_neg,
        max_len,
        neg_min_length,
    ),
    strt_index_query,
    audio_path,
    seg_len,
    h5_handles,
) = sample

print("\n=== Sample Structure ===")
print(f"X_pos (support positive): {type(X_pos)}, shape={X_pos.shape}, dtype={X_pos.dtype}")
print(f"X_neg (support negative): {type(X_neg)}, shape={X_neg.shape}, dtype={X_neg.dtype}")
print(f"X_query_info (query metadata): {type(X_query_info)}, length={len(X_query_info)}")
if len(X_query_info) > 0:
    print(f"  First query metadata: {X_query_info[0]}")
print(f"X_pos_neg: {type(X_pos_neg)}, shape={X_pos_neg.shape}")
print(f"X_neg_neg: {type(X_neg_neg)}, shape={X_neg_neg.shape}")
print(f"X_query_info_neg: {type(X_query_info_neg)}, length={len(X_query_info_neg)}")
print(f"hop_seg: {hop_seg}")
print(f"hop_seg_neg: {hop_seg_neg}")
print(f"max_len: {max_len}")
print(f"neg_min_length: {neg_min_length}")
print(f"strt_index_query: {strt_index_query}")
print(f"audio_path: {os.path.basename(audio_path)}")
print(f"seg_len: {seg_len}")
print(f"h5_handles: {type(h5_handles)}, count={len(h5_handles)}")
if len(h5_handles) > 0:
    print(f"  First handle type: {type(h5_handles[0])}")
    print(f"  First handle shape: {h5_handles[0].shape}")

# Verify lazy loading
print("\n=== Verification ===")
print(f"✓ Support segments are numpy arrays: {isinstance(X_pos, np.ndarray)}")
print(f"✓ Query segments are metadata (list): {isinstance(X_query_info, list)}")
print(f"✓ HDF5 handles returned: {len(h5_handles) > 0}")
print(f"✓ Query metadata contains frame info: {'start_frame' in X_query_info[0] if len(X_query_info) > 0 else False}")

# Force GC and measure
print("\nForcing garbage collection...")
after_gc_mem = force_gc()
print(f"Memory after GC: {after_gc_mem:.2f} MB")
print(f"Memory retained: {after_gc_mem - baseline_mem:.2f} MB")

## Section 4: Test DataLoader Integration

Test that DataLoader works with the streaming implementation.
We'll use a simple collate function that handles the metadata.

In [ ]:
def simple_collate_fn(batch):
    """
    Simple collate function for testing.
    Just returns the batch as-is since we're testing with batch_size=1.
    """
    return batch[0] if len(batch) == 1 else batch

print("Creating DataLoader with batch_size=1...")
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=simple_collate_fn,
    num_workers=0  # Single process for memory tracking
)

print(f"DataLoader created with {len(dataloader)} batches")

# Iterate through first few batches
print("\n=== Iterating through batches ===")
num_batches_to_test = min(3, len(dataloader))
memory_per_batch = []

for i, batch in enumerate(dataloader):
    if i >= num_batches_to_test:
        break
    
    print(f"\n--- Batch {i+1}/{num_batches_to_test} ---")
    mem_before = print_mem(f"Before batch {i+1}")
    
    # Unpack batch
    (
        (
            X_pos,
            X_neg,
            X_query_info,
            X_pos_neg,
            X_neg_neg,
            X_query_info_neg,
            hop_seg,
            hop_seg_neg,
            max_len,
            neg_min_length,
        ),
        strt_index_query,
        audio_path,
        seg_len,
        h5_handles,
    ) = batch
    
    print(f"  Support pos shape: {X_pos.shape}")
    print(f"  Support neg shape: {X_neg.shape}")
    print(f"  Query metadata count: {len(X_query_info)}")
    print(f"  HDF5 handles: {len(h5_handles)}")
    
    mem_after = print_mem(f"After batch {i+1}")
    memory_per_batch.append(mem_after - mem_before)
    print(f"  Memory increase: {mem_after - mem_before:.2f} MB")
    
    # Clean up handles for this batch
    for handle in h5_handles:
        if hasattr(handle, 'file'):
            handle.file.close()
    
    gc.collect()

print("\n=== DataLoader Summary ===")
print(f"Average memory per batch: {np.mean(memory_per_batch):.2f} MB")
print(f"Max memory per batch: {np.max(memory_per_batch):.2f} MB")
print(f"Min memory per batch: {np.min(memory_per_batch):.2f} MB")
print("✓ DataLoader integration successful")
print("✓ No eager loading detected (query segments are metadata)")

## Section 5: Test StreamingSegmentDataset Directly

Test the StreamingSegmentDataset class with actual metadata and handles.

In [ ]:
print("Testing StreamingSegmentDataset...")

# Get a fresh sample
sample = dataset[0]
(
    (
        X_pos,
        X_neg,
        X_query_info,
        X_pos_neg,
        X_neg_neg,
        X_query_info_neg,
        hop_seg,
        hop_seg_neg,
        max_len,
        neg_min_length,
    ),
    strt_index_query,
    audio_path,
    seg_len,
    h5_handles,
) = sample

print(f"Query segments to stream: {len(X_query_info)}")
mem_before_streaming = print_mem("Before StreamingSegmentDataset")

# Create streaming dataset
streaming_dataset = StreamingSegmentDataset(
    h5_handles=h5_handles,
    segments=X_query_info,
    seg_len=seg_len,
    cache_size=10  # Small cache for testing
)

mem_after_create = print_mem("After creating StreamingSegmentDataset")
print(f"Memory increase: {mem_after_create - mem_before_streaming:.2f} MB")
print(f"Dataset length: {len(streaming_dataset)}")

# Load several segments
print("\n=== Loading segments ===")
num_segments_to_test = min(5, len(streaming_dataset))
segment_memories = []

for i in range(num_segments_to_test):
    mem_before_seg = print_mem(f"Before segment {i}")
    
    segment, label = streaming_dataset[i]
    
    mem_after_seg = print_mem(f"After segment {i}")
    segment_memories.append(mem_after_seg - mem_before_seg)
    
    print(f"  Segment {i}: shape={segment.shape}, dtype={segment.dtype}, label={label}")
    print(f"  Memory increase: {mem_after_seg - mem_before_seg:.2f} MB")

# Check cache stats
cache_stats = streaming_dataset.get_cache_stats()
print(f"\n=== Cache Statistics ===")
print(f"Cache size: {cache_stats['cache_size']}/{cache_stats['max_cache_size']}")
print(f"Cache utilization: {cache_stats['cache_utilization']:.2%}")

# Test cache hit
print("\n=== Testing cache hit ===")
mem_before_cache_hit = print_mem("Before cache hit")
segment_cached, _ = streaming_dataset[0]  # Should hit cache
mem_after_cache_hit = print_mem("After cache hit")
print(f"Memory increase (should be ~0): {mem_after_cache_hit - mem_before_cache_hit:.2f} MB")

# Clear cache
print("\n=== Clearing cache ===")
mem_before_clear = print_mem("Before cache clear")
streaming_dataset.clear_cache()
mem_after_clear = print_mem("After cache clear")
print(f"Memory freed: {mem_before_clear - mem_after_clear:.2f} MB")

print("\n✓ StreamingSegmentDataset test complete")
print(f"Average memory per segment: {np.mean(segment_memories):.2f} MB")

# Clean up
for handle in h5_handles:
    if hasattr(handle, 'file'):
        handle.file.close()

## Section 6: Cleanup Tests

Verify that memory is properly released after cleanup.

In [ ]:
print("=== Cleanup Test ===")
mem_before_cleanup = print_mem("Before cleanup")

# Delete all references
print("Deleting dataset and related objects...")
if 'dataset' in locals():
    del dataset
if 'dataloader' in locals():
    del dataloader
if 'streaming_dataset' in locals():
    del streaming_dataset
if 'sample' in locals():
    del sample
if 'batch' in locals():
    del batch

# Force garbage collection
print("Forcing garbage collection...")
gc.collect()
gc.collect()  # Run twice for good measure
gc.collect()

mem_after_cleanup = print_mem("After cleanup")
mem_freed = mem_before_cleanup - mem_after_cleanup
mem_vs_baseline = mem_after_cleanup - baseline_mem

print(f"\n=== Cleanup Summary ===")
print(f"Memory freed: {mem_freed:.2f} MB")
print(f"Memory vs baseline: {mem_vs_baseline:.2f} MB")
print(f"Cleanup efficiency: {(mem_freed / (mem_before_cleanup - baseline_mem) * 100):.1f}%")

if mem_vs_baseline < 50:  # Less than 50MB retained
    print("✓ Excellent cleanup - minimal memory retained")
elif mem_vs_baseline < 100:
    print("✓ Good cleanup - acceptable memory retained")
else:
    print("⚠️  Warning - significant memory retained after cleanup")

# Stop tracemalloc
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"\n=== Tracemalloc Summary ===")
print(f"Current memory: {current / 1024 / 1024:.2f} MB")
print(f"Peak memory: {peak / 1024 / 1024:.2f} MB")

## Summary and Conclusions

### Expected Behavior:

1. **Dataset Instantiation**: Minimal memory increase (only metadata)
2. **`__getitem__()`**: 
   - Support segments loaded as numpy arrays (eager)
   - Query segments returned as metadata (lazy)
   - HDF5 handles remain open
3. **DataLoader**: 
   - Consistent memory per batch
   - No memory spikes from eager loading
4. **StreamingSegmentDataset**:
   - Segments loaded on-demand
   - Cache working correctly
   - Memory freed when cache cleared
5. **Cleanup**:
   - Memory returns close to baseline
   - HDF5 handles properly closed

### Red Flags:

- ❌ Large memory spike during `__getitem__()` (indicates eager loading)
- ❌ Query segments are numpy arrays instead of metadata
- ❌ Memory grows linearly with DataLoader iterations
- ❌ Memory not freed after cleanup
- ❌ HDF5 file handle errors

### Next Steps:

If tests pass:
- ✓ Streaming implementation is working correctly
- ✓ Ready for integration with training pipeline
- ✓ Can proceed with full-scale testing

If tests fail:
- Review implementation in `test_loader_ada_seglen_better_neg_v2.py`
- Check HDF5 file creation and structure
- Verify collate function handles metadata correctly
- Debug memory leaks with profiling tools